
# ot5.ipynb

`ot5`는 ot4의 개선된 싸움/스토킹 판정에 cf6 원본 방식의 SegFormer 인도주행 판정을 실제로 결합한 버전입니다.

핵심 변경점:

1. 모든 결과를 `C:\Users\우성한\outputs\prediction\aa` 폴더 안에 저장합니다.
2. 기존 CF2 객체 탐지 JSON을 생성합니다.
3. 새 중간 JSON인 `aa_features`를 생성합니다.
   - 사람 중심점 변화
   - 간단한 track id
   - 사람 간 거리 변화
   - 근접 상태에서의 움직임
   - 방향 변화
   - 위치 역전
   - bbox overlap
   - 5초 슬라이딩 윈도우 fight/stalking score
4. 최종적으로 `bd4_improved` 행동 판정과 `cf6_like` 이륜차 판정을 동시에 저장합니다.

입력 영상 루트:

```python
C:\Users\우성한\Sample (1)\Sample\01.원천데이터
```

In [1]:
# Cell 1. 경로 및 전체 설정
from pathlib import Path
import os, json, math, time, gc
from collections import Counter, defaultdict

VIDEO_ROOT = Path(r"C:\Users\우성한\Sample (1)\Sample\01.원천데이터")
OUTPUT_ROOT = Path(r"C:\Users\우성한\outputs\prediction")

# 모든 결과는 aa 폴더 아래 저장
AA_DIR = OUTPUT_ROOT / "aa"
AA_CF2_DIR = AA_DIR / "cf2"
AA_FEATURE_DIR = AA_DIR / "aa_features"
AA_BD4_DIR = AA_DIR / "ot5_bd4_improved"
AA_CF6_DIR = AA_DIR / "ot5_cf6_real"
AA_FINAL_DIR = AA_DIR / "final_ot5"
AA_DEBUG_DIR = AA_DIR / "debug"

for d in [AA_DIR, AA_CF2_DIR, AA_FEATURE_DIR, AA_BD4_DIR, AA_CF6_DIR, AA_FINAL_DIR, AA_DEBUG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

VIDEO_EXTS = {
    ".mp4", ".avi", ".mov", ".mkv", ".wmv", ".mpeg", ".mpg", ".m4v",
    ".MP4", ".AVI", ".MOV", ".MKV", ".WMV", ".MPEG", ".MPG", ".M4V"
}

# 테스트할 때만 숫자로 바꾸고, 전체 처리 시 None
MAX_VIDEOS = None
MAX_JSON_FILES = None
MAX_FRAMES_PER_VIDEO = None
# MAX_VIDEOS = 3
# MAX_JSON_FILES = 3
# MAX_FRAMES_PER_VIDEO = 160

YOLO_MODEL_PATH = "yolov8n.pt"
CF2_FRAME_INTERVAL = 10   # 싸움 움직임을 보려면 15보다 10이 조금 유리함
PERSON_MIN_CONF = 0.15
VEHICLE_MIN_CONF = 0.10
TRACK_MAX_DIST = 130
WINDOW_SECONDS = 5.0
WINDOW_STRIDE_SECONDS = 1.5

# 판정 threshold. 결과가 너무 보수적이면 FIGHT_THRESHOLD를 0.34 정도로 낮추고,
# 너무 포괄적이면 0.42 이상으로 올리면 됨.
FIGHT_THRESHOLD = 0.38
STALKING_THRESHOLD = 0.36

TWO_WHEELER_CLASSES = {"bicycle", "motorcycle"}
PERSON_CLASSES = {"person"}

# cf6 real segmentation 설정
SEG_MODEL_ID_GPU = "nvidia/segformer-b5-finetuned-cityscapes-1024-1024"
SEG_MODEL_ID_CPU = "nvidia/segformer-b0-finetuned-cityscapes-1024-1024"
ROAD_ID = 0
SIDEWALK_ID = 1
MASK_PATCH_RADIUS = 16
DILATE_KERNEL_SIZE = 9
BOTTOM_BAND_Y1 = 0.72
BOTTOM_BAND_Y2 = 1.00
BOTTOM_BAND_X_MARGIN = 0.08
SIDEWALK_SCORE_THRESHOLD = 0.18
ROAD_SCORE_THRESHOLD = 0.18
SIDEWALK_MARGIN_OVER_ROAD = 0.03
ROAD_MARGIN_OVER_SIDEWALK = 0.02
USE_AMP_ON_CUDA = True

# 헬멧 모델은 선택 사항. 없으면 helmet_status는 unknown으로 남김.
HELMET_MODEL_PATH = None
HELMET_CONF = 0.15

print("VIDEO_ROOT exists:", VIDEO_ROOT.exists(), VIDEO_ROOT)
print("AA_DIR:", AA_DIR)

if not VIDEO_ROOT.exists():
    raise FileNotFoundError(f"원본 영상 루트를 찾지 못했습니다: {VIDEO_ROOT}")

VIDEO_ROOT exists: True C:\Users\우성한\Sample (1)\Sample\01.원천데이터
AA_DIR: C:\Users\우성한\outputs\prediction\aa


In [2]:
# Cell 2. 의존성 확인
missing = []

try:
    import cv2
    print("cv2 OK")
except Exception as e:
    missing.append("opencv-python")
    print("cv2 import failed:", repr(e))

try:
    import numpy as np
    print("numpy OK")
except Exception as e:
    missing.append("numpy")
    print("numpy import failed:", repr(e))

try:
    from tqdm import tqdm
    print("tqdm OK")
except Exception as e:
    missing.append("tqdm")
    print("tqdm import failed:", repr(e))

try:
    from ultralytics import YOLO
    print("ultralytics OK")
except Exception as e:
    YOLO = None
    missing.append("ultralytics")
    print("ultralytics import failed:", repr(e))

if missing:
    print("\n설치 필요 후보:", sorted(set(missing)))
    print("예: pip install opencv-python numpy tqdm ultralytics")

cv2 OK
numpy OK
tqdm OK
ultralytics OK


In [3]:
# Cell 3. 원천데이터 전체 영상 탐색
video_paths = [p for p in VIDEO_ROOT.rglob("*") if p.is_file() and p.suffix in VIDEO_EXTS]
video_paths = sorted(video_paths)

if MAX_VIDEOS is not None:
    video_paths = video_paths[:MAX_VIDEOS]

print("찾은 영상 수:", len(video_paths))
for p in video_paths[:30]:
    print(p)

if len(video_paths) == 0:
    raise RuntimeError("원천데이터 경로 안에서 영상 파일을 찾지 못했습니다.")

찾은 영상 수: 22
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 구역 내 지속 배회\lo_e1281\lo_e1281_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 구역 내 지속 배회\lo_e1281\lo_e1281_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 인물을 뒤따라가며 배회\fp_e1520\fp_e1520_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 인물을 뒤따라가며 배회\fp_e1520\fp_e1520_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\싸움\신체적 충돌을 동반한 싸움\ph_e2063\ph_e2063_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\싸움\신체적 충돌을 동반한 싸움\ph_e2063\ph_e2063_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\오토바이 인도 주행\ms_e3500\ms_e3500_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\오토바이 인도 주행\ms_e3500\ms_e3500_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\자전거 인도 주행\bs_e4387\bs_e4387_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\자전거 인도 주행\bs_e4387\bs_e4387_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\전동킥보드 인도 주행\es_e3768\es_e3768_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데

In [4]:
# Cell 4. CF2 생성/확인
if YOLO is None:
    raise RuntimeError("ultralytics YOLO를 사용할 수 없습니다.")

cf2_yolo_model = YOLO(YOLO_MODEL_PATH)
print("YOLO model loaded:", YOLO_MODEL_PATH)


def safe_stem_from_path(path: Path, root: Path):
    rel = path.relative_to(root)
    return "__".join(rel.with_suffix("").parts)


def video_meta(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return {"fps": None, "width": None, "height": None, "frame_count": None}
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return {
        "fps": float(fps) if fps and fps > 0 else None,
        "width": width,
        "height": height,
        "frame_count": frame_count,
    }


def create_cf2_from_video(video_path, output_json_path, frame_interval=10):
    meta = video_meta(video_path)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"영상 열기 실패: {video_path}")

    frames = []
    frame_idx = 0
    sampled_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if MAX_FRAMES_PER_VIDEO is not None and frame_idx >= MAX_FRAMES_PER_VIDEO:
            break

        if frame_idx % frame_interval == 0:
            result = cf2_yolo_model(frame, verbose=False)[0]
            objects = []
            classes = {}

            if result.boxes is not None:
                for box in result.boxes:
                    cls_id = int(box.cls[0])
                    cls_name = cf2_yolo_model.names[cls_id]
                    conf = float(box.conf[0])
                    bbox = [round(float(v), 2) for v in box.xyxy[0].tolist()]
                    objects.append({"class": cls_name, "confidence": round(conf, 4), "bbox": bbox})
                    classes[cls_name] = classes.get(cls_name, 0) + 1

            timestamp_sec = None
            if meta.get("fps"):
                timestamp_sec = round(frame_idx / meta["fps"], 3)

            frames.append({
                "frame_id": frame_idx,
                "sampled_id": sampled_idx,
                "timestamp_sec": timestamp_sec,
                "object_count": len(objects),
                "classes": classes,
                "objects": objects,
            })
            sampled_idx += 1

        frame_idx += 1

    cap.release()

    summary_classes = {}
    for fr in frames:
        for cls_name, count in fr["classes"].items():
            summary_classes[cls_name] = summary_classes.get(cls_name, 0) + count

    data = {
        "video_name": video_path.name,
        "video_path": str(video_path),
        "relative_video_path": str(video_path.relative_to(VIDEO_ROOT)),
        "meta": {
            **meta,
            "cf2_frame_interval": frame_interval,
            "sampled_frames": len(frames),
            "actually_read_frames": frame_idx,
        },
        "summary": {
            "total_frames_read": frame_idx,
            "sampled_frames": len(frames),
            "classes": summary_classes,
        },
        "frames": frames,
    }

    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    return data


created, used_existing, failed = 0, 0, []

for video_path in tqdm(video_paths):
    stem = safe_stem_from_path(video_path, VIDEO_ROOT)
    output_json_path = AA_CF2_DIR / f"{stem}.json"

    if output_json_path.exists():
        used_existing += 1
        continue

    try:
        create_cf2_from_video(video_path, output_json_path, frame_interval=CF2_FRAME_INTERVAL)
        created += 1
    except Exception as e:
        failed.append({"video": str(video_path), "error": repr(e)})

print("\n================================")
print("AA CF2 생성/확인 완료")
print("전체 영상:", len(video_paths))
print("생성:", created)
print("기존 사용:", used_existing)
print("실패:", len(failed))
print("저장 위치:", AA_CF2_DIR)
print("================================")

if failed:
    print("\n실패 목록 일부:")
    for item in failed[:20]:
        print(item)

YOLO model loaded: yolov8n.pt


100%|██████████████████████████████████████████████████████████████████████████████████| 22/22 [02:13<00:00,  6.06s/it]


AA CF2 생성/확인 완료
전체 영상: 22
생성: 22
기존 사용: 0
실패: 0
저장 위치: C:\Users\우성한\outputs\prediction\aa\cf2


In [5]:
# Cell 5. aa_features 생성용 함수

def clamp(v, lo=0.0, hi=1.0):
    return max(lo, min(hi, v))


def bbox_center(bbox):
    x1, y1, x2, y2 = bbox
    return ((x1 + x2) / 2, (y1 + y2) / 2)


def bbox_area(bbox):
    x1, y1, x2, y2 = bbox
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def bbox_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    union = bbox_area(a) + bbox_area(b) - inter
    return inter / union if union > 0 else 0.0


def dist(p1, p2):
    return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)


def vec(p1, p2):
    return (p2[0] - p1[0], p2[1] - p1[1])


def vec_norm(v):
    return math.sqrt(v[0] ** 2 + v[1] ** 2)


def cosine(v1, v2):
    n1, n2 = vec_norm(v1), vec_norm(v2)
    if n1 < 1e-6 or n2 < 1e-6:
        return None
    return (v1[0] * v2[0] + v1[1] * v2[1]) / (n1 * n2)


def extract_person_detections(frame):
    persons = []
    for obj in frame.get("objects", []):
        if obj.get("class") in PERSON_CLASSES and obj.get("confidence", 0) >= PERSON_MIN_CONF:
            bbox = obj.get("bbox")
            if bbox and len(bbox) == 4:
                persons.append({
                    "bbox": bbox,
                    "center": bbox_center(bbox),
                    "area": bbox_area(bbox),
                    "confidence": obj.get("confidence", 0),
                })
    return persons


def extract_vehicle_detections(frame):
    vehicles = []
    for obj in frame.get("objects", []):
        if obj.get("class") in TWO_WHEELER_CLASSES and obj.get("confidence", 0) >= VEHICLE_MIN_CONF:
            bbox = obj.get("bbox")
            if bbox and len(bbox) == 4:
                vehicles.append({
                    "class": obj.get("class"),
                    "bbox": bbox,
                    "center": bbox_center(bbox),
                    "area": bbox_area(bbox),
                    "confidence": obj.get("confidence", 0),
                })
    return vehicles


def assign_tracks(frames):
    """간단한 greedy tracker. 외부 tracking 라이브러리 없이 person bbox 흐름만 대략 연결."""
    next_track_id = 1
    active = {}  # track_id -> last center
    active_last_seen = {}
    tracked_frames = []

    for idx, frame in enumerate(frames):
        persons = extract_person_detections(frame)
        used_tracks = set()

        # 각 person을 가장 가까운 기존 track에 배정
        for p in persons:
            best_tid, best_d = None, None
            for tid, center in active.items():
                if tid in used_tracks:
                    continue
                if idx - active_last_seen.get(tid, -999) > 2:
                    continue
                d = dist(p["center"], center)
                if d <= TRACK_MAX_DIST and (best_d is None or d < best_d):
                    best_tid, best_d = tid, d

            if best_tid is None:
                best_tid = next_track_id
                next_track_id += 1

            p["track_id"] = best_tid
            p["track_match_dist"] = round(best_d, 3) if best_d is not None else None
            active[best_tid] = p["center"]
            active_last_seen[best_tid] = idx
            used_tracks.add(best_tid)

        vehicles = extract_vehicle_detections(frame)
        tracked_frames.append({
            "frame_id": frame.get("frame_id"),
            "sampled_id": frame.get("sampled_id"),
            "timestamp_sec": frame.get("timestamp_sec"),
            "persons": persons,
            "vehicles": vehicles,
            "person_count": len(persons),
            "vehicle_count": len(vehicles),
        })

    return tracked_frames


def add_motion_features(tracked_frames):
    last_by_track = {}
    prev_velocity = {}

    for fr in tracked_frames:
        for p in fr["persons"]:
            tid = p["track_id"]
            prev = last_by_track.get(tid)

            if prev is None:
                p["motion"] = 0.0
                p["velocity"] = [0.0, 0.0]
                p["direction_change"] = 0.0
            else:
                v = vec(prev["center"], p["center"])
                m = vec_norm(v)
                old_v = prev_velocity.get(tid)
                c = cosine(old_v, v) if old_v is not None else None
                # direction_change: 0이면 방향 유지, 1이면 반대/급변에 가까움
                dc = 0.0 if c is None else clamp((1 - c) / 2)
                p["motion"] = round(m, 3)
                p["velocity"] = [round(v[0], 3), round(v[1], 3)]
                p["direction_change"] = round(dc, 3)
                prev_velocity[tid] = v

            last_by_track[tid] = {"center": p["center"], "frame_id": fr["frame_id"]}

    return tracked_frames


def pair_key(a, b):
    return tuple(sorted([a, b]))


def compute_pair_interactions(tracked_frames, width=None, height=None):
    """프레임별 사람쌍 상호작용 특징 생성."""
    diag = math.sqrt((width or 1920) ** 2 + (height or 1080) ** 2)
    close_dist = diag * 0.075  # 대략 1920x1080 기준 165px 근처
    near_dist = diag * 0.12

    prev_pair_info = {}

    for fr in tracked_frames:
        persons = fr["persons"]
        interactions = []

        for i in range(len(persons)):
            for j in range(i + 1, len(persons)):
                a, b = persons[i], persons[j]
                d = dist(a["center"], b["center"])
                iou = bbox_iou(a["bbox"], b["bbox"])
                avg_motion = (a.get("motion", 0) + b.get("motion", 0)) / 2
                max_motion = max(a.get("motion", 0), b.get("motion", 0))
                avg_dir_change = (a.get("direction_change", 0) + b.get("direction_change", 0)) / 2

                key = pair_key(a["track_id"], b["track_id"])
                prev = prev_pair_info.get(key)
                distance_delta = 0.0
                approach_or_separate = 0
                position_swap = 0
                same_direction = 0

                if prev is not None:
                    distance_delta = d - prev["distance"]
                    if abs(distance_delta) > close_dist * 0.20:
                        approach_or_separate = 1

                    # x축 상대 위치가 바뀌면 서로 뒤섞였을 가능성
                    prev_order = prev["x_order"]
                    now_order = a["center"][0] < b["center"][0]
                    if prev_order != now_order and d < near_dist:
                        position_swap = 1

                    ca = cosine(a.get("velocity", [0, 0]), b.get("velocity", [0, 0]))
                    if ca is not None and ca > 0.65:
                        same_direction = 1

                close_flag = int(d <= close_dist)
                near_flag = int(d <= near_dist)
                close_motion_flag = int(close_flag and avg_motion > close_dist * 0.12)
                overlap_flag = int(iou > 0.015)

                item = {
                    "track_pair": list(key),
                    "distance": round(d, 3),
                    "distance_norm": round(d / diag, 5),
                    "distance_delta": round(distance_delta, 3),
                    "close": close_flag,
                    "near": near_flag,
                    "iou": round(iou, 4),
                    "overlap": overlap_flag,
                    "avg_motion": round(avg_motion, 3),
                    "max_motion": round(max_motion, 3),
                    "avg_direction_change": round(avg_dir_change, 3),
                    "close_motion": close_motion_flag,
                    "approach_or_separate": approach_or_separate,
                    "position_swap": position_swap,
                    "same_direction": same_direction,
                }
                interactions.append(item)

                prev_pair_info[key] = {
                    "distance": d,
                    "x_order": a["center"][0] < b["center"][0],
                }

        fr["interactions"] = interactions
        fr["interaction_count"] = len(interactions)
        fr["close_pair_count"] = sum(x["close"] for x in interactions)
        fr["close_motion_pair_count"] = sum(x["close_motion"] for x in interactions)
        fr["overlap_pair_count"] = sum(x["overlap"] for x in interactions)
        fr["position_swap_count"] = sum(x["position_swap"] for x in interactions)
        fr["same_direction_pair_count"] = sum(x["same_direction"] for x in interactions)

    return tracked_frames


def window_indices(n, window_n, stride_n):
    if n <= 0:
        return []
    window_n = max(2, min(window_n, n))
    stride_n = max(1, stride_n)
    out = []
    start = 0
    while start < n:
        end = min(n, start + window_n)
        out.append((start, end))
        if end == n:
            break
        start += stride_n
    return out


def window_score(items):
    n = len(items)
    if n == 0:
        return {}

    multi_rate = sum(1 for f in items if f["person_count"] >= 2) / n
    person_rate = sum(1 for f in items if f["person_count"] >= 1) / n
    close_rate = sum(1 for f in items if f["close_pair_count"] > 0) / n
    close_motion_rate = sum(1 for f in items if f["close_motion_pair_count"] > 0) / n
    overlap_rate = sum(1 for f in items if f["overlap_pair_count"] > 0) / n
    swap_rate = sum(1 for f in items if f["position_swap_count"] > 0) / n
    same_dir_rate = sum(1 for f in items if f["same_direction_pair_count"] > 0) / n
    vehicle_rate = sum(1 for f in items if f["vehicle_count"] > 0) / n

    # 프레임 내 사람들의 평균 움직임/방향변화
    motions = []
    dir_changes = []
    for f in items:
        for p in f["persons"]:
            motions.append(p.get("motion", 0))
            dir_changes.append(p.get("direction_change", 0))
    avg_motion = sum(motions) / len(motions) if motions else 0.0
    avg_dir_change = sum(dir_changes) / len(dir_changes) if dir_changes else 0.0

    # 접근/분리 반복성
    approach_sep_rate = sum(
        1 for f in items if any(x.get("approach_or_separate") for x in f.get("interactions", []))
    ) / n

    # 싸움: 근접 + 움직임 + 방향 급변 + 위치 뒤섞임 중심
    fight_raw = (
        0.12 * multi_rate +
        0.14 * close_rate +
        0.24 * close_motion_rate +
        0.16 * approach_sep_rate +
        0.14 * clamp(avg_dir_change / 0.45) +
        0.10 * overlap_rate +
        0.10 * swap_rate
    )

    # 같은 방향으로 안정적으로 이동하면 fight보다는 stalking/동행에 가까움
    stable_follow_penalty = 0.14 * same_dir_rate if close_motion_rate < 0.35 else 0.05 * same_dir_rate
    fight_score = clamp(fight_raw - stable_follow_penalty)

    # 스토킹: 사람 지속 + 같은 방향 이동 + 거리 안정 + fight 요소는 낮음
    stalking_raw = (
        0.25 * person_rate +
        0.20 * multi_rate +
        0.20 * same_dir_rate +
        0.15 * approach_sep_rate +
        0.10 * clamp(avg_motion / 80.0) +
        0.10 * (1.0 - overlap_rate)
    )
    stalking_penalty = 0.20 * close_motion_rate + 0.10 * swap_rate
    stalking_score = clamp(stalking_raw - stalking_penalty)

    return {
        "frames": n,
        "person_rate": round(person_rate, 4),
        "multi_rate": round(multi_rate, 4),
        "close_rate": round(close_rate, 4),
        "close_motion_rate": round(close_motion_rate, 4),
        "overlap_rate": round(overlap_rate, 4),
        "swap_rate": round(swap_rate, 4),
        "same_direction_rate": round(same_dir_rate, 4),
        "approach_separate_rate": round(approach_sep_rate, 4),
        "vehicle_rate": round(vehicle_rate, 4),
        "avg_motion": round(avg_motion, 4),
        "avg_direction_change": round(avg_dir_change, 4),
        "fight_score": round(fight_score, 4),
        "stalking_score": round(stalking_score, 4),
    }


def build_aa_features(cf2_data):
    frames = cf2_data.get("frames", [])
    meta = cf2_data.get("meta", {})
    fps = meta.get("fps") or 30.0
    frame_interval = meta.get("cf2_frame_interval") or CF2_FRAME_INTERVAL
    width = meta.get("width") or 1920
    height = meta.get("height") or 1080

    tracked_frames = assign_tracks(frames)
    tracked_frames = add_motion_features(tracked_frames)
    tracked_frames = compute_pair_interactions(tracked_frames, width=width, height=height)

    samples_per_second = fps / frame_interval if frame_interval else 2.0
    window_n = max(3, int(round(WINDOW_SECONDS * samples_per_second)))
    stride_n = max(1, int(round(WINDOW_STRIDE_SECONDS * samples_per_second)))

    windows = []
    for start, end in window_indices(len(tracked_frames), window_n, stride_n):
        ws = window_score(tracked_frames[start:end])
        if not ws:
            continue
        start_ts = tracked_frames[start].get("timestamp_sec")
        end_ts = tracked_frames[end - 1].get("timestamp_sec")
        windows.append({
            "start_index": start,
            "end_index": end - 1,
            "start_frame_id": tracked_frames[start].get("frame_id"),
            "end_frame_id": tracked_frames[end - 1].get("frame_id"),
            "start_sec": start_ts,
            "end_sec": end_ts,
            **ws,
        })

    peak_fight = max(windows, key=lambda x: x.get("fight_score", 0), default=None)
    peak_stalking = max(windows, key=lambda x: x.get("stalking_score", 0), default=None)

    overall = window_score(tracked_frames)

    data = {
        "video_name": cf2_data.get("video_name"),
        "video_path": cf2_data.get("video_path"),
        "relative_video_path": cf2_data.get("relative_video_path"),
        "meta": {
            **meta,
            "window_seconds": WINDOW_SECONDS,
            "window_stride_seconds": WINDOW_STRIDE_SECONDS,
            "window_n_sampled_frames": window_n,
            "stride_n_sampled_frames": stride_n,
            "fight_threshold": FIGHT_THRESHOLD,
            "stalking_threshold": STALKING_THRESHOLD,
        },
        "overall_features": overall,
        "peak_fight_window": peak_fight,
        "peak_stalking_window": peak_stalking,
        "windows": windows,
        "frames": tracked_frames,
    }
    return data

In [6]:
# Cell 6. aa_features 전체 생성
cf2_json_paths = sorted(AA_CF2_DIR.rglob("*.json"))
if MAX_JSON_FILES is not None:
    cf2_json_paths = cf2_json_paths[:MAX_JSON_FILES]

print("CF2 json 수:", len(cf2_json_paths))
if len(cf2_json_paths) == 0:
    raise RuntimeError("AA_CF2_DIR 안에 CF2 json이 없습니다. Cell 4를 먼저 확인하세요.")

created, used_existing, failed = 0, 0, []

for json_path in tqdm(cf2_json_paths):
    out_path = AA_FEATURE_DIR / json_path.name
    if out_path.exists():
        used_existing += 1
        continue
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            cf2_data = json.load(f)
        aa_data = build_aa_features(cf2_data)
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(aa_data, f, ensure_ascii=False, indent=2)
        created += 1
    except Exception as e:
        failed.append({"json": str(json_path), "error": repr(e)})

print("\n================================")
print("aa_features 생성/확인 완료")
print("전체 json:", len(cf2_json_paths))
print("생성:", created)
print("기존 사용:", used_existing)
print("실패:", len(failed))
print("저장 위치:", AA_FEATURE_DIR)
print("================================")

if failed:
    print("\n실패 목록 일부:")
    for item in failed[:20]:
        print(item)

CF2 json 수: 22


100%|██████████████████████████████████████████████████████████████████████████████████| 22/22 [00:00<00:00, 69.68it/s]


aa_features 생성/확인 완료
전체 json: 22
생성: 22
기존 사용: 0
실패: 0
저장 위치: C:\Users\우성한\outputs\prediction\aa\aa_features


In [7]:
# Cell 7. improved bd4 행동 판정 함수

def classify_action_from_aa(aa_data):
    peak_fight = aa_data.get("peak_fight_window") or {}
    peak_stalking = aa_data.get("peak_stalking_window") or {}
    overall = aa_data.get("overall_features") or {}

    fight_score = peak_fight.get("fight_score", 0.0)
    stalking_score = peak_stalking.get("stalking_score", 0.0)

    # fight는 peak window 중심으로 판단한다.
    # 단, 사람이 2명 이상 거의 없으면 fight 제외.
    fight_multi_ok = peak_fight.get("multi_rate", 0) >= 0.25
    fight_interaction_ok = (
        peak_fight.get("close_motion_rate", 0) >= 0.18 or
        peak_fight.get("approach_separate_rate", 0) >= 0.25 or
        peak_fight.get("swap_rate", 0) >= 0.12 or
        peak_fight.get("overlap_rate", 0) >= 0.12
    )

    # stalking은 fight보다 움직임이 덜 뒤섞이고, 같은 방향성이 어느 정도 있어야 함.
    stalking_person_ok = peak_stalking.get("person_rate", 0) >= 0.35
    stalking_not_fight_like = peak_stalking.get("close_motion_rate", 0) < 0.45 and peak_stalking.get("swap_rate", 0) < 0.25

    if fight_score >= FIGHT_THRESHOLD and fight_multi_ok and fight_interaction_ok:
        label = "fight"
        reason = (
            "영상 전체 평균이 아니라 가장 강한 짧은 구간에서, "
            "근접 상태의 움직임/접근-분리/방향 변화/위치 뒤섞임이 함께 나타나 싸움 후보로 판단"
        )
    elif stalking_score >= STALKING_THRESHOLD and stalking_person_ok and stalking_not_fight_like:
        label = "stalking"
        reason = (
            "사람이 지속적으로 등장하고, 싸움처럼 격한 근접 움직임보다는 "
            "이동 흐름 또는 따라가는 형태가 더 강해 스토킹 후보로 판단"
        )
    else:
        label = "other"
        reason = (
            "짧은 구간 기준 peak score를 보아도 fight/stalking 조건을 충분히 만족하지 않아 기타로 판단"
        )

    return {
        "label": label,
        "reason": reason,
        "scores": {
            "peak_fight_score": round(fight_score, 4),
            "peak_stalking_score": round(stalking_score, 4),
            "overall_fight_score": overall.get("fight_score", 0),
            "overall_stalking_score": overall.get("stalking_score", 0),
        },
        "peak_fight_window": peak_fight,
        "peak_stalking_window": peak_stalking,
        "overall_features": overall,
    }

In [8]:
# Cell 8. improved bd4 행동 판정 실행 및 저장
aa_json_paths = sorted(AA_FEATURE_DIR.rglob("*.json"))
if MAX_JSON_FILES is not None:
    aa_json_paths = aa_json_paths[:MAX_JSON_FILES]

bd4_results = []
bd4_failed = []

for json_path in tqdm(aa_json_paths):
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            aa_data = json.load(f)
        pred = classify_action_from_aa(aa_data)
        item = {
            "video_name": aa_data.get("video_name"),
            "video_path": aa_data.get("video_path"),
            "relative_video_path": aa_data.get("relative_video_path"),
            **pred,
        }
        bd4_results.append(item)
        out_path = AA_BD4_DIR / json_path.name
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(item, f, ensure_ascii=False, indent=2)
    except Exception as e:
        bd4_failed.append({"json": str(json_path), "error": repr(e)})

bd4_summary = {
    "task": "ot5_bd4_improved_action_classification",
    "input_dir": str(AA_FEATURE_DIR),
    "output_dir": str(AA_BD4_DIR),
    "total": len(aa_json_paths),
    "success": len(bd4_results),
    "failed": len(bd4_failed),
    "label_counts": dict(Counter(r["label"] for r in bd4_results)),
    "results": bd4_results,
    "failed_items": bd4_failed,
}

bd4_summary_path = AA_BD4_DIR / "ot5_bd4_improved_summary.json"
with open(bd4_summary_path, "w", encoding="utf-8") as f:
    json.dump(bd4_summary, f, ensure_ascii=False, indent=2)

print("\n================================")
print("ot5 bd4 improved 판정 완료")
print("전체:", bd4_summary["total"])
print("성공:", bd4_summary["success"])
print("실패:", bd4_summary["failed"])
print("label_counts:", bd4_summary["label_counts"])
print("summary 저장:", bd4_summary_path)
print("================================")

100%|█████████████████████████████████████████████████████████████████████████████████| 22/22 [00:00<00:00, 191.83it/s]


ot5 bd4 improved 판정 완료
전체: 22
성공: 22
실패: 0
label_counts: {'stalking': 9, 'fight': 4, 'other': 9}
summary 저장: C:\Users\우성한\outputs\prediction\aa\ot5_bd4_improved\ot5_bd4_improved_summary.json


In [9]:
# Cell 9. cf6 원본 방식 포함 여부 확인 + SegFormer/헬멧 모델 로드
# 이 셀의 목적:
# 1) cf6의 핵심인 SegFormer road/sidewalk segmentation이 실제로 포함되어 있는지 확인
# 2) 헬멧 모델은 선택 사항으로 연결
# 3) 아래 셀에서 sidewalk_status가 unknown만 나오는지 점검 가능하게 함

import cv2
import numpy as np
from PIL import Image

CF6_CAPABILITY_REPORT = {
    "real_cf6_sidewalk_segmentation_code_included": True,
    "segformer_loaded": False,
    "segformer_model_id": None,
    "helmet_model_loaded": False,
    "helmet_model_path": None,
    "helmet_backend": "none",
}

try:
    import torch
    from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    SEG_MODEL_ID = SEG_MODEL_ID_GPU if DEVICE.type == "cuda" else SEG_MODEL_ID_CPU
    print("DEVICE:", DEVICE)
    print("SEG_MODEL_ID:", SEG_MODEL_ID)
    if DEVICE.type != "cuda":
        print("경고: CUDA가 아니므로 SegFormer b0로 실행합니다. 전체 처리 시간이 길 수 있습니다.")

    seg_processor = SegformerImageProcessor.from_pretrained(SEG_MODEL_ID)
    seg_model = SegformerForSemanticSegmentation.from_pretrained(SEG_MODEL_ID)
    seg_model.to(DEVICE)
    seg_model.eval()

    CF6_CAPABILITY_REPORT["segformer_loaded"] = True
    CF6_CAPABILITY_REPORT["segformer_model_id"] = SEG_MODEL_ID
    print("SegFormer loaded:", SEG_MODEL_ID)
except Exception as e:
    DEVICE = None
    seg_processor = None
    seg_model = None
    print("SegFormer 로드 실패. 인도주행 판정은 unknown이 됩니다:", repr(e))

helmet_model = None
helmet_backend = "none"

try:
    if HELMET_MODEL_PATH is not None and YOLO is not None:
        helmet_path = Path(HELMET_MODEL_PATH)
        if helmet_path.exists():
            helmet_model = YOLO(str(helmet_path))
            try:
                if DEVICE is not None:
                    helmet_model.to(str(DEVICE))
            except Exception:
                pass
            helmet_backend = "local_yolo"
            CF6_CAPABILITY_REPORT["helmet_model_loaded"] = True
            CF6_CAPABILITY_REPORT["helmet_model_path"] = str(helmet_path)
            CF6_CAPABILITY_REPORT["helmet_backend"] = helmet_backend
            print("helmet model loaded:", helmet_path)
        else:
            print("HELMET_MODEL_PATH가 설정됐지만 파일이 없습니다:", helmet_path)
    else:
        print("HELMET_MODEL_PATH is None -> 헬멧은 unknown으로 처리")
except Exception as e:
    print("helmet model load failed:", repr(e))

print("CF6_CAPABILITY_REPORT:")
print(json.dumps(CF6_CAPABILITY_REPORT, ensure_ascii=False, indent=2))


# Cell 10. cf6 원본 방식의 bbox/rider matching + road/sidewalk/helmet 함수

def bottom_center(bbox):
    x1, y1, x2, y2 = bbox
    return ((x1 + x2) / 2, y2)


def intersection_area(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    x1 = max(ax1, bx1); y1 = max(ay1, by1)
    x2 = min(ax2, bx2); y2 = min(ay2, by2)
    return max(0, x2 - x1) * max(0, y2 - y1)


def extract_objects_cf6(frame):
    persons, vehicles = [], []
    for obj in frame.get("objects", []):
        cls = obj.get("class")
        conf = float(obj.get("confidence", 0.0))
        bbox = obj.get("bbox")
        if not bbox or len(bbox) != 4:
            continue
        clean = {
            "class": cls,
            "confidence": round(conf, 4),
            "bbox": [int(round(v)) for v in bbox],
        }
        if cls in {"person", "rider"} and conf >= PERSON_MIN_CONF:
            persons.append(clean)
        elif cls in TWO_WHEELER_CLASSES and conf >= VEHICLE_MIN_CONF:
            vehicles.append(clean)
    return persons, vehicles


def match_riders_cf6(persons, vehicles):
    pairs = []
    used_persons = set()
    for vehicle_idx, vehicle in enumerate(vehicles):
        vb = vehicle["bbox"]
        vcenter = bbox_center(vb)
        best = None
        for person_idx, person in enumerate(persons):
            if person_idx in used_persons:
                continue
            pb = person["bbox"]
            pc = bbox_center(pb)
            d = dist(pc, vcenter)
            inter = intersection_area(pb, vb)
            overlap = inter / max(1, min(bbox_area(pb), bbox_area(vb)))
            vertical_ok = pc[1] <= vb[3] + 100
            close_ok = d <= 280
            overlap_ok = overlap >= 0.015
            if not vertical_ok or not (close_ok or overlap_ok):
                continue
            score = overlap * 2.0 + max(0, 1 - d / 280)
            if best is None or score > best["match_score"]:
                best = {
                    "person_index": person_idx,
                    "vehicle_index": vehicle_idx,
                    "person": person,
                    "vehicle": vehicle,
                    "distance": round(d, 2),
                    "overlap": round(overlap, 4),
                    "match_score": round(score, 4),
                }
        if best is not None:
            used_persons.add(best["person_index"])
            pairs.append(best)
    return pairs


def read_frame_from_cap(cap, frame_id: int):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_id))
    ret, frame_bgr = cap.read()
    if not ret:
        return None
    return cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)


def segment_frame(image_rgb: np.ndarray):
    if seg_model is None or seg_processor is None:
        return None
    pil_image = Image.fromarray(image_rgb)
    inputs = seg_processor(images=pil_image, return_tensors="pt")
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        if DEVICE.type == "cuda" and USE_AMP_ON_CUDA:
            with torch.cuda.amp.autocast():
                outputs = seg_model(**inputs)
        else:
            outputs = seg_model(**inputs)
    logits = outputs.logits
    upsampled_logits = torch.nn.functional.interpolate(
        logits,
        size=image_rgb.shape[:2],
        mode="bilinear",
        align_corners=False,
    )
    pred = upsampled_logits.argmax(dim=1)[0].detach().cpu().numpy().astype(np.uint8)
    return pred


def make_class_masks(pred):
    road = (pred == ROAD_ID).astype(np.uint8)
    sidewalk = (pred == SIDEWALK_ID).astype(np.uint8)
    if DILATE_KERNEL_SIZE and DILATE_KERNEL_SIZE > 1:
        k = np.ones((DILATE_KERNEL_SIZE, DILATE_KERNEL_SIZE), np.uint8)
        road_d = cv2.dilate(road, k, iterations=1).astype(bool)
        sidewalk_d = cv2.dilate(sidewalk, k, iterations=1).astype(bool)
    else:
        road_d = road.astype(bool)
        sidewalk_d = sidewalk.astype(bool)
    return road.astype(bool), sidewalk.astype(bool), road_d, sidewalk_d


def safe_crop_mask(mask, x1, y1, x2, y2):
    h, w = mask.shape[:2]
    x1 = max(0, min(w, int(round(x1))))
    x2 = max(0, min(w, int(round(x2))))
    y1 = max(0, min(h, int(round(y1))))
    y2 = max(0, min(h, int(round(y2))))
    if x1 >= x2 or y1 >= y2:
        return None
    return mask[y1:y2, x1:x2]


def ratio_in_box(mask, bbox):
    crop = safe_crop_mask(mask, *bbox)
    if crop is None or crop.size == 0:
        return 0.0
    return float(crop.mean())


def mask_patch_ratio(mask, center, radius=16):
    cx, cy = center
    crop = safe_crop_mask(mask, cx - radius, cy - radius, cx + radius + 1, cy + radius + 1)
    if crop is None or crop.size == 0:
        return 0.0
    return float(crop.mean())


def expand_bbox(bbox, w_img, h_img, scale_x=0.12, scale_y=0.10):
    x1, y1, x2, y2 = bbox
    bw, bh = x2 - x1, y2 - y1
    nx1 = max(0, x1 - bw * scale_x)
    nx2 = min(w_img, x2 + bw * scale_x)
    ny1 = max(0, y1 - bh * scale_y)
    ny2 = min(h_img, y2 + bh * scale_y)
    return [nx1, ny1, nx2, ny2]


def union_bbox(a, b):
    return [min(a[0], b[0]), min(a[1], b[1]), max(a[2], b[2]), max(a[3], b[3])]


def bottom_band_bbox(bbox):
    x1, y1, x2, y2 = bbox
    bw, bh = x2 - x1, y2 - y1
    bx1 = x1 + bw * BOTTOM_BAND_X_MARGIN
    bx2 = x2 - bw * BOTTOM_BAND_X_MARGIN
    by1 = y1 + bh * BOTTOM_BAND_Y1
    by2 = y1 + bh * BOTTOM_BAND_Y2
    return [bx1, by1, bx2, by2]


def judge_sidewalk_v2(pred, person_bbox, vehicle_bbox):
    if pred is None:
        return "unknown", "SegFormer not loaded", 0.0, 0.0

    road, sidewalk, road_d, sidewalk_d = make_class_masks(pred)
    h, w = pred.shape[:2]
    vehicle_expanded = expand_bbox(vehicle_bbox, w, h, 0.10, 0.08)
    person_vehicle_union = expand_bbox(union_bbox(person_bbox, vehicle_bbox), w, h, 0.06, 0.05)
    vehicle_bottom = bottom_band_bbox(vehicle_bbox)
    bc = bottom_center(vehicle_bbox)

    sidewalk_scores = [
        ratio_in_box(sidewalk_d, vehicle_expanded),
        ratio_in_box(sidewalk_d, vehicle_bottom),
        ratio_in_box(sidewalk_d, person_vehicle_union),
        mask_patch_ratio(sidewalk_d, bc, MASK_PATCH_RADIUS),
    ]
    road_scores = [
        ratio_in_box(road_d, vehicle_expanded),
        ratio_in_box(road_d, vehicle_bottom),
        ratio_in_box(road_d, person_vehicle_union),
        mask_patch_ratio(road_d, bc, MASK_PATCH_RADIUS),
    ]
    sidewalk_score = max(sidewalk_scores)
    road_score = max(road_scores)

    if sidewalk_score >= SIDEWALK_SCORE_THRESHOLD and sidewalk_score >= road_score + SIDEWALK_MARGIN_OVER_ROAD:
        return "sidewalk_riding", f"sidewalk_score={sidewalk_score:.3f} > road_score={road_score:.3f}", road_score, sidewalk_score
    if sidewalk_score >= SIDEWALK_SCORE_THRESHOLD * 0.75 and sidewalk_score >= road_score - 0.02:
        return "sidewalk_candidate", f"sidewalk candidate: sidewalk_score={sidewalk_score:.3f}, road_score={road_score:.3f}", road_score, sidewalk_score
    if road_score >= ROAD_SCORE_THRESHOLD and road_score >= sidewalk_score + ROAD_MARGIN_OVER_SIDEWALK:
        return "road_riding", f"road_score={road_score:.3f} > sidewalk_score={sidewalk_score:.3f}", road_score, sidewalk_score
    return "unknown", f"weak road/sidewalk evidence: road_score={road_score:.3f}, sidewalk_score={sidewalk_score:.3f}", road_score, sidewalk_score


def crop_head_and_upper_body(image_rgb, person_bbox, mode="head"):
    h_img, w_img = image_rgb.shape[:2]
    x1, y1, x2, y2 = [int(round(v)) for v in person_bbox]
    bw, bh = x2 - x1, y2 - y1
    if bw <= 0 or bh <= 0:
        return None
    if mode == "head":
        cy2 = y1 + int(bh * 0.45)
        mx = int(bw * 0.35); my = int(bh * 0.12)
    else:
        cy2 = y1 + int(bh * 0.62)
        mx = int(bw * 0.25); my = int(bh * 0.08)
    cx1 = max(0, x1 - mx)
    cx2 = min(w_img, x2 + mx)
    cy1 = max(0, y1 - my)
    cy2 = min(h_img, cy2 + my)
    if cx1 >= cx2 or cy1 >= cy2:
        return None
    return image_rgb[cy1:cy2, cx1:cx2]


def normalize_helmet_label(label):
    s = str(label).lower().replace("-", "_").replace(" ", "_")
    if any(k in s for k in ["no_helmet", "without", "nohelmet", "not_helmet", "bare"]):
        return "no_helmet"
    if "helmet" in s or "hardhat" in s:
        return "helmet"
    return "unknown"


def judge_helmet(image_rgb, person_bbox):
    if helmet_model is None:
        return "unknown", "helmet model is not connected", []
    all_detections = []
    for mode in ["head", "upper"]:
        crop = crop_head_and_upper_body(image_rgb, person_bbox, mode=mode)
        if crop is None:
            continue
        try:
            preds = helmet_model(crop, verbose=False, conf=HELMET_CONF)
        except Exception as e:
            return "unknown", f"helmet predict failed: {repr(e)}", []
        for res in preds:
            names = getattr(res, "names", {})
            boxes = getattr(res, "boxes", None)
            if boxes is None:
                continue
            for b in boxes:
                cls_id = int(b.cls[0].item())
                conf = float(b.conf[0].item())
                label = names.get(cls_id, str(cls_id)) if isinstance(names, dict) else str(cls_id)
                norm = normalize_helmet_label(label)
                all_detections.append({"label": label, "normalized": norm, "confidence": round(conf, 4), "crop_mode": mode})
        recognized = [d for d in all_detections if d["normalized"] in {"helmet", "no_helmet"}]
        if recognized:
            best_no = max([d for d in recognized if d["normalized"] == "no_helmet"], key=lambda x: x["confidence"], default=None)
            best_hel = max([d for d in recognized if d["normalized"] == "helmet"], key=lambda x: x["confidence"], default=None)
            if best_no and (best_hel is None or best_no["confidence"] >= best_hel["confidence"] - 0.05):
                return "no_helmet", f"detected no_helmet conf={best_no['confidence']}", all_detections
            if best_hel:
                return "helmet", f"detected helmet conf={best_hel['confidence']}", all_detections
    return "unknown", "no recognized helmet/no_helmet detection", all_detections


# Cell 11. cf6 real 이륜차/인도주행/헬멧 판정 실행 및 영상 단위 요약 저장

def classify_cf6_real_for_video(cf2_data):
    video_name = cf2_data.get("video_name")
    video_path = Path(cf2_data.get("video_path"))
    frames = cf2_data.get("frames", [])

    frame_results = []
    failed_reason = None

    if not video_path.exists():
        return {
            "video_name": video_name,
            "video_path": str(video_path),
            "relative_video_path": cf2_data.get("relative_video_path"),
            "label": "no_two_wheeler",
            "tags": [],
            "sidewalk_status": "unknown",
            "helmet_status": "unknown",
            "two_wheeler_detected": False,
            "rider_detected": False,
            "cf6_real_included": True,
            "segformer_loaded": CF6_CAPABILITY_REPORT.get("segformer_loaded", False),
            "frame_results": [],
            "features": {"failed_reason": f"video file not found: {video_path}"},
        }

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return {
            "video_name": video_name,
            "video_path": str(video_path),
            "relative_video_path": cf2_data.get("relative_video_path"),
            "label": "no_two_wheeler",
            "tags": [],
            "sidewalk_status": "unknown",
            "helmet_status": "unknown",
            "two_wheeler_detected": False,
            "rider_detected": False,
            "cf6_real_included": True,
            "segformer_loaded": CF6_CAPABILITY_REPORT.get("segformer_loaded", False),
            "frame_results": [],
            "features": {"failed_reason": "cv2.VideoCapture failed"},
        }

    try:
        for frame in frames:
            frame_id = int(frame.get("frame_id", -1))
            persons, vehicles = extract_objects_cf6(frame)
            if not vehicles:
                continue
            pairs = match_riders_cf6(persons, vehicles) if persons else []
            if not pairs:
                # 이륜차만 있는 경우도 감지 기록
                frame_results.append({
                    "frame_id": frame_id,
                    "pair_index": None,
                    "sidewalk_status": "unknown",
                    "helmet_status": "unknown",
                    "road_score": 0.0,
                    "sidewalk_score": 0.0,
                    "anomaly_types": [],
                    "description": "이륜차 감지, rider 매칭 불확실",
                    "vehicle_only": True,
                })
                continue

            image_rgb = read_frame_from_cap(cap, frame_id)
            pred = None
            if image_rgb is not None and seg_model is not None:
                try:
                    pred = segment_frame(image_rgb)
                except RuntimeError as e:
                    if DEVICE is not None and DEVICE.type == "cuda" and "out of memory" in str(e).lower():
                        torch.cuda.empty_cache()
                    pred = None
                except Exception:
                    pred = None

            for pair_idx, pair in enumerate(pairs):
                person = pair["person"]
                vehicle = pair["vehicle"]
                sidewalk_status, sidewalk_reason, road_score, sidewalk_score = judge_sidewalk_v2(
                    pred, person_bbox=person["bbox"], vehicle_bbox=vehicle["bbox"]
                )
                if image_rgb is not None:
                    helmet_status, helmet_reason, helmet_detections = judge_helmet(image_rgb, person["bbox"])
                else:
                    helmet_status, helmet_reason, helmet_detections = "unknown", "frame read failed", []

                anomaly_types = []
                if sidewalk_status in {"sidewalk_riding", "sidewalk_candidate"}:
                    anomaly_types.append(sidewalk_status)
                if helmet_status == "no_helmet":
                    anomaly_types.append("no_helmet")

                frame_results.append({
                    "frame_id": frame_id,
                    "pair_index": pair_idx,
                    "person": person,
                    "vehicle": vehicle,
                    "match": {"score": pair["match_score"], "overlap": pair["overlap"], "distance": pair["distance"]},
                    "sidewalk_status": sidewalk_status,
                    "sidewalk_reason": sidewalk_reason,
                    "road_score": round(float(road_score), 4),
                    "sidewalk_score": round(float(sidewalk_score), 4),
                    "helmet_status": helmet_status,
                    "helmet_reason": helmet_reason,
                    "helmet_detections": helmet_detections,
                    "anomaly_types": anomaly_types,
                    "description": f"sidewalk={sidewalk_status}, helmet={helmet_status}",
                    "vehicle_only": False,
                })
    finally:
        cap.release()

    sidewalk_counter = Counter(r.get("sidewalk_status", "unknown") for r in frame_results)
    helmet_counter = Counter(r.get("helmet_status", "unknown") for r in frame_results)
    anomaly_counter = Counter(t for r in frame_results for t in r.get("anomaly_types", []))

    two_wheeler_detected = len(frame_results) > 0
    rider_detected = any(not r.get("vehicle_only") for r in frame_results)

    if sidewalk_counter.get("sidewalk_riding", 0) > 0:
        sidewalk_status = "sidewalk_riding"
    elif sidewalk_counter.get("sidewalk_candidate", 0) > 0:
        sidewalk_status = "sidewalk_candidate"
    elif sidewalk_counter.get("road_riding", 0) > 0:
        sidewalk_status = "road_riding"
    else:
        sidewalk_status = "unknown"

    if helmet_counter.get("no_helmet", 0) > 0:
        helmet_status = "no_helmet"
    elif helmet_counter.get("helmet", 0) > 0:
        helmet_status = "helmet"
    else:
        helmet_status = "unknown"

    tags = []
    if two_wheeler_detected:
        tags.append("two_wheeler_detected")
    if rider_detected:
        tags.append("rider_like")
    if sidewalk_status in {"sidewalk_riding", "sidewalk_candidate", "road_riding"}:
        tags.append(sidewalk_status)
    if helmet_status == "no_helmet":
        tags.append("no_helmet")

    if two_wheeler_detected and sidewalk_status == "sidewalk_riding" and helmet_status == "no_helmet":
        label = "two_wheeler_sidewalk_and_no_helmet"
    elif two_wheeler_detected and sidewalk_status == "sidewalk_riding":
        label = "two_wheeler_sidewalk_riding"
    elif two_wheeler_detected and sidewalk_status == "sidewalk_candidate":
        label = "two_wheeler_sidewalk_candidate"
    elif two_wheeler_detected and helmet_status == "no_helmet":
        label = "two_wheeler_no_helmet"
    elif two_wheeler_detected and rider_detected:
        label = "two_wheeler_rider_like"
    elif two_wheeler_detected:
        label = "two_wheeler_detected"
    else:
        label = "no_two_wheeler"

    return {
        "video_name": video_name,
        "video_path": str(video_path),
        "relative_video_path": cf2_data.get("relative_video_path"),
        "label": label,
        "tags": sorted(set(tags)),
        "two_wheeler_detected": two_wheeler_detected,
        "rider_detected": rider_detected,
        "sidewalk_status": sidewalk_status,
        "helmet_status": helmet_status,
        "cf6_real_included": True,
        "segformer_loaded": CF6_CAPABILITY_REPORT.get("segformer_loaded", False),
        "helmet_backend": helmet_backend,
        "features": {
            "frame_result_count": len(frame_results),
            "sidewalk_status_counts": dict(sidewalk_counter),
            "helmet_status_counts": dict(helmet_counter),
            "anomaly_type_counts": dict(anomaly_counter),
        },
        "frame_results": frame_results,
        "reason": "cf6 원본 방식의 rider matching + SegFormer road/sidewalk segmentation + 선택적 helmet 모델로 영상 단위 이륜차 상태를 요약",
    }


cf6_results = []
cf6_failed = []

for json_path in tqdm(cf2_json_paths):
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            cf2_data = json.load(f)
        pred = classify_cf6_real_for_video(cf2_data)
        cf6_results.append(pred)
        out_path = AA_CF6_DIR / json_path.name
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(pred, f, ensure_ascii=False, indent=2)
    except Exception as e:
        cf6_failed.append({"json": str(json_path), "error": repr(e)})

cf6_summary = {
    "task": "ot5_cf6_real_two_wheeler_sidewalk_helmet_classification",
    "capability_report": CF6_CAPABILITY_REPORT,
    "input_dir": str(AA_CF2_DIR),
    "output_dir": str(AA_CF6_DIR),
    "total": len(cf2_json_paths),
    "success": len(cf6_results),
    "failed": len(cf6_failed),
    "label_counts": dict(Counter(r["label"] for r in cf6_results)),
    "sidewalk_status_counts": dict(Counter(r.get("sidewalk_status", "unknown") for r in cf6_results)),
    "helmet_status_counts": dict(Counter(r.get("helmet_status", "unknown") for r in cf6_results)),
    "tag_counts": dict(Counter(t for r in cf6_results for t in r.get("tags", []))),
    "results": cf6_results,
    "failed_items": cf6_failed,
}

cf6_summary_path = AA_CF6_DIR / "ot5_cf6_real_summary.json"
with open(cf6_summary_path, "w", encoding="utf-8") as f:
    json.dump(cf6_summary, f, ensure_ascii=False, indent=2)

print("\n================================")
print("ot5 cf6 real 이륜차/인도주행/헬멧 판정 완료")
print("전체:", cf6_summary["total"])
print("성공:", cf6_summary["success"])
print("실패:", cf6_summary["failed"])
print("label_counts:", cf6_summary["label_counts"])
print("sidewalk_status_counts:", cf6_summary["sidewalk_status_counts"])
print("helmet_status_counts:", cf6_summary["helmet_status_counts"])
print("tag_counts:", cf6_summary["tag_counts"])
print("summary 저장:", cf6_summary_path)
print("================================")

if not CF6_CAPABILITY_REPORT.get("segformer_loaded"):
    print("주의: SegFormer가 로드되지 않아 sidewalk_status가 unknown 위주로 나올 수 있습니다.")
elif cf6_summary["success"] > 0 and set(cf6_summary["sidewalk_status_counts"].keys()) <= {"unknown"}:
    print("주의: SegFormer는 로드됐지만 영상 단위 sidewalk_status가 모두 unknown입니다. rider 매칭 프레임이 적거나 segmentation 기준이 보수적일 수 있습니다.")


# Cell 12. ot5 최종 병합: unknown final_label 방지 + cf6 포함 여부 점검 포함
bd4_by_rel = {r.get("relative_video_path"): r for r in bd4_results}
cf6_by_rel = {r.get("relative_video_path"): r for r in cf6_results}
all_keys = sorted(set(bd4_by_rel) | set(cf6_by_rel))


def valid_label(x):
    if x is None:
        return "unknown"
    x = str(x).strip()
    return x if x else "unknown"


def make_ot5_final_label(bd4_label, cf6_label):
    bd4_label = valid_label(bd4_label)
    cf6_label = valid_label(cf6_label)
    parts = []
    if bd4_label not in {"unknown", "missing_bd4"}:
        parts.append(bd4_label)
    if cf6_label not in {"unknown", "missing_cf6", "no_two_wheeler"}:
        parts.append(cf6_label)
    if not parts:
        parts.append("other")
    return " + ".join(parts)

final_results = []
for rel in all_keys:
    bd4 = bd4_by_rel.get(rel)
    cf6 = cf6_by_rel.get(rel)
    bd4_label = valid_label(bd4.get("label") if bd4 else "missing_bd4")
    cf6_label = valid_label(cf6.get("label") if cf6 else "missing_cf6")
    combined_tags = []
    if bd4_label not in {"unknown", "missing_bd4"}:
        combined_tags.append(bd4_label)
    if cf6:
        combined_tags.extend(cf6.get("tags", []))
        if cf6.get("sidewalk_status") not in {None, "unknown"}:
            combined_tags.append(cf6.get("sidewalk_status"))
        if cf6.get("helmet_status") == "no_helmet":
            combined_tags.append("no_helmet")

    final_label = make_ot5_final_label(bd4_label, cf6_label)
    final_reason_parts = []
    if bd4:
        final_reason_parts.append(f"행동 판정={bd4_label}")
    if cf6:
        final_reason_parts.append(f"이륜차 판정={cf6_label}")
        final_reason_parts.append(f"인도/차도={cf6.get('sidewalk_status', 'unknown')}")
        final_reason_parts.append(f"헬멧={cf6.get('helmet_status', 'unknown')}")
    if not final_reason_parts:
        final_reason_parts.append("유효한 판정 결과가 없어 other로 보정")

    item = {
        "video_name": (bd4 or cf6 or {}).get("video_name"),
        "video_path": (bd4 or cf6 or {}).get("video_path"),
        "relative_video_path": rel,
        "final_label": final_label,
        "parallel_label": final_label,
        "combined_tags": sorted(set(combined_tags)),
        "bd4_label": bd4_label,
        "cf6_label": cf6_label,
        "sidewalk_status": cf6.get("sidewalk_status", "unknown") if cf6 else "missing_cf6",
        "helmet_status": cf6.get("helmet_status", "unknown") if cf6 else "missing_cf6",
        "cf6_real_included": bool(cf6 and cf6.get("cf6_real_included")),
        "segformer_loaded": bool(cf6 and cf6.get("segformer_loaded")),
        "final_reason": " / ".join(final_reason_parts),
        "bd4_action": bd4,
        "cf6_two_wheeler": cf6,
    }
    final_results.append(item)

final_unknown_count = sum(1 for r in final_results if r.get("final_label") in {None, "", "unknown"})
cf6_included_count = sum(1 for r in final_results if r.get("cf6_real_included"))
seg_loaded_count = sum(1 for r in final_results if r.get("segformer_loaded"))

final_summary = {
    "task": "ot5_parallel_action_and_cf6_real_sidewalk_helmet_final_merge",
    "video_root": str(VIDEO_ROOT),
    "aa_dir": str(AA_DIR),
    "cf2_dir": str(AA_CF2_DIR),
    "aa_feature_dir": str(AA_FEATURE_DIR),
    "bd4_dir": str(AA_BD4_DIR),
    "cf6_dir": str(AA_CF6_DIR),
    "final_dir": str(AA_FINAL_DIR),
    "capability_report": CF6_CAPABILITY_REPORT,
    "total": len(final_results),
    "final_unknown_count": final_unknown_count,
    "cf6_included_count": cf6_included_count,
    "segformer_loaded_video_count": seg_loaded_count,
    "bd4_label_counts": dict(Counter(r["bd4_label"] for r in final_results)),
    "cf6_label_counts": dict(Counter(r["cf6_label"] for r in final_results)),
    "final_label_counts": dict(Counter(r["final_label"] for r in final_results)),
    "sidewalk_status_counts": dict(Counter(r["sidewalk_status"] for r in final_results)),
    "helmet_status_counts": dict(Counter(r["helmet_status"] for r in final_results)),
    "combined_tag_counts": dict(Counter(t for r in final_results for t in r.get("combined_tags", []))),
    "results": final_results,
}

final_summary_path = AA_FINAL_DIR / "ot5_final_summary.json"
with open(final_summary_path, "w", encoding="utf-8") as f:
    json.dump(final_summary, f, ensure_ascii=False, indent=2)

print("\n================================")
print("ot5 최종 병합 완료")
print("전체:", final_summary["total"])
print("final_unknown_count:", final_summary["final_unknown_count"])
print("cf6_included_count:", final_summary["cf6_included_count"])
print("segformer_loaded_video_count:", final_summary["segformer_loaded_video_count"])
print("bd4_label_counts:", final_summary["bd4_label_counts"])
print("cf6_label_counts:", final_summary["cf6_label_counts"])
print("final_label_counts:", final_summary["final_label_counts"])
print("sidewalk_status_counts:", final_summary["sidewalk_status_counts"])
print("helmet_status_counts:", final_summary["helmet_status_counts"])
print("summary 저장:", final_summary_path)
print("================================")

if final_unknown_count > 0:
    print("주의: final_label unknown이 남았습니다. 아래 readable 셀에서 해당 항목을 확인하세요.")


# Cell 13. 결과를 알아보기 쉽게 정리하는 셀
summary_path = AA_FINAL_DIR / "ot5_final_summary.json"
with open(summary_path, "r", encoding="utf-8") as f:
    summary = json.load(f)
results = summary.get("results", [])

print("=" * 80)
print("OT5 최종 결과 요약")
print("=" * 80)
print("전체 영상 수:", len(results))
print("final_unknown_count:", summary.get("final_unknown_count"))
print("cf6 포함 영상 수:", summary.get("cf6_included_count"))
print("SegFormer 로드 영상 수:", summary.get("segformer_loaded_video_count"))
print("summary 파일:", summary_path)

print("\n" + "=" * 80)
print("1. 최종 라벨별 개수")
print("=" * 80)
for label, count in Counter(r.get("final_label", "unknown") for r in results).most_common():
    print(f"{label:45s} : {count}")

print("\n" + "=" * 80)
print("2. 행동 판정별 개수")
print("=" * 80)
for label, count in Counter(r.get("bd4_label", "unknown") for r in results).most_common():
    print(f"{label:45s} : {count}")

print("\n" + "=" * 80)
print("3. 이륜차/인도주행 판정별 개수")
print("=" * 80)
print("cf6_label:")
for label, count in Counter(r.get("cf6_label", "unknown") for r in results).most_common():
    print(f"  {label:43s} : {count}")
print("sidewalk_status:")
for label, count in Counter(r.get("sidewalk_status", "unknown") for r in results).most_common():
    print(f"  {label:43s} : {count}")
print("helmet_status:")
for label, count in Counter(r.get("helmet_status", "unknown") for r in results).most_common():
    print(f"  {label:43s} : {count}")

print("\n" + "=" * 80)
print("4. 영상별 최종 판정")
print("=" * 80)
for idx, r in enumerate(results, start=1):
    print("\n" + "-" * 80)
    print(f"[{idx}] {r.get('video_name')}")
    print("상대 경로:", r.get("relative_video_path"))
    print("최종 판정:", r.get("final_label"))
    print("행동 판정:", r.get("bd4_label"))
    print("이륜차 판정:", r.get("cf6_label"))
    print("인도/차도 판정:", r.get("sidewalk_status"))
    print("헬멧 판정:", r.get("helmet_status"))
    print("cf6 포함:", r.get("cf6_real_included"), "/ SegFormer:", r.get("segformer_loaded"))
    print("태그:", ", ".join(r.get("combined_tags", [])) if r.get("combined_tags") else "없음")
    bd4 = r.get("bd4_action") or {}
    scores = bd4.get("scores") or {}
    if scores:
        print("싸움 점수:", scores.get("peak_fight_score"), "/ 스토킹 점수:", scores.get("peak_stalking_score"))
    print("이유:", r.get("final_reason"))

print("\n" + "=" * 80)
print("5. 인도주행/후보 영상만 따로 출력")
print("=" * 80)
sidewalk_hits = [r for r in results if r.get("sidewalk_status") in {"sidewalk_riding", "sidewalk_candidate"}]
if not sidewalk_hits:
    print("인도주행 또는 인도주행 후보 없음")
else:
    for i, r in enumerate(sidewalk_hits, start=1):
        print(f"{i}. {r.get('video_name')} | {r.get('sidewalk_status')} | {r.get('relative_video_path')} | final={r.get('final_label')}")

print("\n" + "=" * 80)
print("6. final_label이 unknown인 항목 점검")
print("=" * 80)
unknowns = [r for r in results if r.get("final_label") in {None, "", "unknown"}]
if not unknowns:
    print("unknown 최종 판정 없음")
else:
    for r in unknowns:
        print(r.get("video_name"), r.get("relative_video_path"), r.get("bd4_label"), r.get("cf6_label"))

readable_path = AA_FINAL_DIR / "ot5_readable_result.txt"
lines = []
lines.append("OT5 최종 결과 요약")
lines.append("=" * 80)
lines.append(f"전체 영상 수: {len(results)}")
lines.append(f"final_unknown_count: {summary.get('final_unknown_count')}")
lines.append("")
lines.append("[최종 라벨별 개수]")
for label, count in Counter(r.get("final_label", "unknown") for r in results).most_common():
    lines.append(f"{label:45s} : {count}")
lines.append("")
lines.append("[영상별 최종 판정]")
for idx, r in enumerate(results, start=1):
    lines.append("-" * 80)
    lines.append(f"[{idx}] {r.get('video_name')}")
    lines.append(f"상대 경로: {r.get('relative_video_path')}")
    lines.append(f"최종 판정: {r.get('final_label')}")
    lines.append(f"행동 판정: {r.get('bd4_label')}")
    lines.append(f"이륜차 판정: {r.get('cf6_label')}")
    lines.append(f"인도/차도 판정: {r.get('sidewalk_status')}")
    lines.append(f"헬멧 판정: {r.get('helmet_status')}")
    lines.append(f"태그: {', '.join(r.get('combined_tags', []))}")
    lines.append(f"이유: {r.get('final_reason')}")
with open(readable_path, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))
print("\nTXT 요약 저장 완료:", readable_path)


def search_ot5_results(query, limit=30):
    q = str(query).lower()
    aliases = {
        "싸움": ["fight"],
        "스토킹": ["stalking"],
        "인도": ["sidewalk_riding", "sidewalk_candidate"],
        "보도": ["sidewalk_riding", "sidewalk_candidate"],
        "차도": ["road_riding"],
        "헬멧": ["helmet", "no_helmet"],
        "미착용": ["no_helmet"],
        "이륜차": ["two_wheeler"],
        "오토바이": ["motorcycle", "two_wheeler"],
        "자전거": ["bicycle", "two_wheeler"],
    }
    keys = []
    for k, vals in aliases.items():
        if k in q:
            keys.extend(vals)
    if not keys:
        keys = [q]
    hits = []
    for r in results:
        hay = json.dumps(r, ensure_ascii=False).lower()
        if all(any(key in hay for key in [kk]) for kk in keys):
            hits.append(r)
    print("QUERY:", query, "hits:", len(hits))
    for r in hits[:limit]:
        print("-", r.get("video_name"), "|", r.get("final_label"), "|", r.get("sidewalk_status"), "|", r.get("relative_video_path"))
    return hits

print("\n검색 예시:")
print('search_ot5_results("인도 주행 이륜차")')
print('search_ot5_results("싸움")')
print('search_ot5_results("스토킹")')

DEVICE: cuda
SEG_MODEL_ID: nvidia/segformer-b5-finetuned-cityscapes-1024-1024


Loading weights:   0%|          | 0/1172 [00:00<?, ?it/s]

SegFormer loaded: nvidia/segformer-b5-finetuned-cityscapes-1024-1024
HELMET_MODEL_PATH is None -> 헬멧은 unknown으로 처리
CF6_CAPABILITY_REPORT:
{
  "real_cf6_sidewalk_segmentation_code_included": true,
  "segformer_loaded": true,
  "segformer_model_id": "nvidia/segformer-b5-finetuned-cityscapes-1024-1024",
  "helmet_model_loaded": false,
  "helmet_model_path": null,
  "helmet_backend": "none"
}


  0%|                                                                                           | 0/22 [00:00<?, ?it/s]C:\Users\우성한\AppData\Local\Temp\ipykernel_26748\3376372619.py:161: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████████████████████████████████████████████████████████████████████████████| 22/22 [00:41<00:00,  1.90s/it]


ot5 cf6 real 이륜차/인도주행/헬멧 판정 완료
전체: 22
성공: 22
실패: 0
label_counts: {'two_wheeler_rider_like': 8, 'no_two_wheeler': 11, 'two_wheeler_sidewalk_riding': 2, 'two_wheeler_detected': 1}
sidewalk_status_counts: {'road_riding': 8, 'unknown': 12, 'sidewalk_riding': 2}
helmet_status_counts: {'unknown': 22}
tag_counts: {'rider_like': 10, 'road_riding': 8, 'two_wheeler_detected': 11, 'sidewalk_riding': 2}
summary 저장: C:\Users\우성한\outputs\prediction\aa\ot5_cf6_real\ot5_cf6_real_summary.json

ot5 최종 병합 완료
전체: 22
final_unknown_count: 0
cf6_included_count: 22
segformer_loaded_video_count: 22
bd4_label_counts: {'stalking': 9, 'fight': 4, 'other': 9}
cf6_label_counts: {'two_wheeler_rider_like': 8, 'no_two_wheeler': 11, 'two_wheeler_sidewalk_riding': 2, 'two_wheeler_detected': 1}
final_label_counts: {'stalking + two_wheeler_rider_like': 5, 'fight': 3, 'stalking': 2, 'other + two_wheeler_rider_like': 2, 'other': 6, 'other + two_wheeler_sidewalk_riding': 1, 'stalking + two_wheeler_detected': 1, 'fight + two